# Advanced NLP — Phase 2: Hybrid RAG

Dense retrieval, handwritten BM25, and Reciprocal Rank Fusion (RRF). The notebook also includes title/sentence retrieval, a gold-masked test, and official evaluation.

**Kaggle:** Attach `Phase2_Kaggle_Input` and `Phase2_Source_Kaggle`, enable GPU, and run the cells in order. Set `LIMIT_QUESTIONS = 3` for a short test or `None` for the complete run.

## 1. Setup

Install dependencies and load experiment settings from `configs/phase2.yaml`.


In [ ]:
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'transformers>=5.17.0,<6', 'accelerate>=1.7.0,<2',
    'bitsandbytes>=0.46.0', 'pyyaml>=6.0', 'safetensors>=0.5.0'
])
print('Dependencies ready.')

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from functools import partial
import gc
import hashlib
import json
import os
import random
import re
import shutil
import sys
import zipfile

import numpy as np
import yaml

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

ON_KAGGLE = Path('/kaggle/input').is_dir()
ROOT = Path('/kaggle/working/advanced_nlp_phase2') if ON_KAGGLE else Path.cwd() / 'advanced_nlp_phase2'
SOURCE_DIR = ROOT / 'input'
OUTPUT_DIR = ROOT / 'runs_phase2_fair'
REPORT_DIR = ROOT / 'reports_fair'
CACHE_DIR = ROOT / 'cache'
for folder in (SOURCE_DIR, OUTPUT_DIR, REPORT_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)


### Load YAML settings

The YAML file is the source of truth for the model, retrievers, generation, and run settings.


In [ ]:
def find_config():
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        for folder in (base, base / 'Phase2'):
            path = folder / 'configs' / 'phase2.yaml'
            if path.is_file():
                return path

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for path in sorted(kaggle_input.rglob('phase2.yaml')):
            return path
        for archive_path in sorted(kaggle_input.rglob('*.zip')):
            if not zipfile.is_zipfile(archive_path):
                continue
            with zipfile.ZipFile(archive_path) as archive:
                names = [name for name in archive.namelist()
                         if name.endswith('configs/phase2.yaml')]
                if names:
                    path = ROOT / 'configs' / 'phase2.yaml'
                    path.parent.mkdir(parents=True, exist_ok=True)
                    path.write_bytes(archive.read(names[0]))
                    return path
    raise FileNotFoundError('Attach the Phase2_Source_Config_Kaggle dataset, including configs/phase2.yaml.')

CONFIG_PATH = find_config()
CFG = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print('Configuration:', CONFIG_PATH)


In [ ]:
SEED = int(CFG['seed'])
random.seed(SEED)
np.random.seed(SEED)

EMBEDDER_NAME = CFG['models']['embedder']
LLM_NAME = CFG['models']['llm']['name']
LLM_QUANTIZATION = CFG['models']['llm']['quantization']
LLM_DTYPE = CFG['models']['llm']['compute_dtype']
QUERY_INSTRUCTION = CFG['embedding']['query_instruction']
BM25_K1 = float(CFG['retrieval']['bm25_k1'])
BM25_B = float(CFG['retrieval']['bm25_b'])
RRF_K = int(CFG['retrieval']['rrf_k'])
RETRIEVED_K = int(CFG['retrieval']['final_k'])
CANDIDATE_OPTIONS = tuple(CFG['retrieval']['candidate_options'])
MAX_CANDIDATES = max(CANDIDATE_OPTIONS)
CONTEXT_TOKEN_BUDGET = int(CFG['generation']['max_context_tokens'])
MAX_INPUT_TOKENS = int(CFG['generation']['max_input_tokens'])
MAX_NEW_TOKENS = int(CFG['generation']['max_new_tokens'])
DO_SAMPLE = bool(CFG['generation']['do_sample'])
ENABLE_THINKING = bool(CFG['generation']['enable_thinking'])
EMBED_MAX_LENGTH = int(CFG['embedding']['max_length'])
EMBED_BATCH_SIZE = int(CFG['embedding']['batch_size'])
LIMIT_QUESTIONS = CFG['execution']['limit_questions']
GENERATE_GRANULARITY_ANSWERS = CFG['execution']['generate_granularity_answers']
print('Model:', LLM_NAME, '| BM25:', BM25_K1, BM25_B, '| RRF:', RRF_K)


## 2. Shared source code

Core algorithms live in `src/phase2_core.py`, not in notebook cells. This section finds and imports the source package.

In [ ]:
def locate_source():
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for path in sorted(kaggle_input.rglob('phase2_core.py')):
            if path.parent.name == 'src':
                return path.parent.parent

    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        for folder in (base, base / 'Phase2'):
            if (folder / 'src' / 'phase2_core.py').is_file():
                return folder

    if kaggle_input.is_dir():
        for path in sorted(kaggle_input.rglob('*.zip')):
            with zipfile.ZipFile(path) as archive:
                matches = [name for name in archive.namelist()
                           if name.endswith('src/phase2_core.py')]
                if not matches:
                    continue
                prefix = matches[0][:-len('src/phase2_core.py')]
                target = ROOT / 'shared_source'
                (target / 'src').mkdir(parents=True, exist_ok=True)
                for name in ('__init__.py', 'phase2_core.py', 'phase2_engineering.py'):
                    member = prefix + 'src/' + name
                    if member in archive.namelist():
                        (target / 'src' / name).write_bytes(archive.read(member))
                return target

    raise FileNotFoundError('Add the Phase2_Source_Kaggle dataset to Kaggle Inputs.')

In [ ]:
SOURCE_CODE_ROOT = locate_source()
if str(SOURCE_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_CODE_ROOT))

from src.phase2_core import (
    BM25, bm25_tokens, make_units, rrf_fusion,
    parse_grounded_output, passage_block,
    pack_context as core_pack_context,
    cosine_rank as core_cosine_rank,
)
import src.phase2_core as shared_core

CORE_PATH = Path(shared_core.__file__).resolve()
CORE_MODULE_SHA256 = hashlib.sha256(CORE_PATH.read_bytes()).hexdigest()
print('Core module:', CORE_PATH)

## 3. Load the course data

Load the corpus, development questions, hidden questions, and Dev labels. Gold labels are never used to tune the hidden split.

In [ ]:
NEEDED_FILES = [
    'course_pack_v1/corpus.jsonl',
    'course_pack_v1/questions_dev.jsonl',
    'course_pack_v1/questions_hidden.jsonl',
    'course_pack_v1/labels_dev.jsonl',
    'score.py',
]

def input_is_ready(folder):
    return all((folder / name).is_file() for name in NEEDED_FILES)

if not input_is_ready(SOURCE_DIR):
    roots = [Path('/kaggle/input')] if ON_KAGGLE else [Path.cwd(), Path.cwd().parent, Path('/mnt/data')]
    archives = [path for root in roots if root.is_dir()
                for path in root.rglob('Phase2_Kaggle_Input.zip')]
    if archives:
        with zipfile.ZipFile(archives[0]) as archive:
            if any(name.startswith('/') or '..' in Path(name).parts for name in archive.namelist()):
                raise ValueError('Invalid path in the input archive.')
            archive.extractall(SOURCE_DIR)
    else:
        candidates = [path.parent.parent for root in roots if root.is_dir()
                      for path in root.rglob('course_pack_v1/corpus.jsonl')]
        matches = [folder for folder in candidates if input_is_ready(folder)]
        if not matches:
            raise FileNotFoundError('Add Phase2_Kaggle_Input to Kaggle Inputs.')
        shutil.copytree(matches[0] / 'course_pack_v1', SOURCE_DIR / 'course_pack_v1',
                        dirs_exist_ok=True)
        shutil.copy2(matches[0] / 'score.py', SOURCE_DIR / 'score.py')

In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

PACK = SOURCE_DIR / 'course_pack_v1'
corpus = read_jsonl(PACK / 'corpus.jsonl')
questions = {split: read_jsonl(PACK / f'questions_{split}.jsonl')
             for split in ('dev', 'hidden')}
labels_dev = {row['qid']: row for row in read_jsonl(PACK / 'labels_dev.jsonl')}
corpus_by_id = {row['doc_id']: row for row in corpus}

print(f"Corpus: {len(corpus)} | Dev: {len(questions['dev'])} | Hidden: {len(questions['hidden'])}")

## 4. Build BM25 indexes

Use the same handwritten BM25 implementation for passage, title, and sentence units.

In [ ]:
bm25_indexes = {
    mode: BM25(make_units(corpus, mode), BM25_K1, BM25_B)
    for mode in ('passage', 'title', 'sentence')
}
for mode, index in bm25_indexes.items():
    print(f'{mode}: {index.N} indexed units')

## 5. Compute or reuse dense embeddings

Previously saved embeddings are reused when their model and question IDs match.

In [ ]:
EMBED_CACHE_DEV = CACHE_DIR / 'qwen3_embedding4b_masked_dev.npz'
EMBED_CACHE_HIDDEN = CACHE_DIR / 'qwen3_embedding4b_hidden_queries.npz'

for target in (EMBED_CACHE_DEV, EMBED_CACHE_HIDDEN):
    if ON_KAGGLE and not target.exists():
        matches = list(Path('/kaggle/input').rglob(target.name))
        if matches:
            shutil.copy2(matches[0], target)
            print('Using input cache:', target.name)

In [ ]:
def cache_ready(path, split):
    if not path.is_file():
        return False
    with np.load(path, allow_pickle=False) as data:
        expected = [q['qid'] for q in questions[split]]
        if data['qids'].tolist() != expected:
            raise ValueError(f'Cached question IDs do not match: {path.name}')
        if (str(data['checkpoint'].item()) != EMBEDDER_NAME or
                int(data['max_length'].item()) != EMBED_MAX_LENGTH):
            raise ValueError(f'Cached model settings do not match: {path.name}')
        if split == 'dev' and data['doc_ids'].tolist() != [p['doc_id'] for p in corpus]:
            raise ValueError('Cached document IDs do not match the corpus.')
    return True

In [ ]:
def last_token_pool(last_hidden, attention_mask):
    import torch
    if bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item()):
        return last_hidden[:, -1]
    positions = attention_mask.sum(dim=1) - 1
    return last_hidden[torch.arange(last_hidden.shape[0], device=last_hidden.device), positions]


def encode_embeddings(texts, label, model, tokenizer, device):
    import torch
    import torch.nn.functional as F
    vectors = []
    with torch.inference_mode():
        for start in range(0, len(texts), EMBED_BATCH_SIZE):
            batch = texts[start:start + EMBED_BATCH_SIZE]
            inputs = tokenizer(batch, return_tensors='pt', padding=True,
                               truncation=True, max_length=EMBED_MAX_LENGTH).to(device)
            result = model(**inputs)
            pooled = last_token_pool(result.last_hidden_state, inputs['attention_mask'])
            vectors.append(F.normalize(pooled.float(), p=2, dim=-1).cpu().numpy())
            del result, pooled, inputs
            if (start // EMBED_BATCH_SIZE) % 100 == 0:
                print(label, min(start + len(batch), len(texts)), '/', len(texts))
    return np.concatenate(vectors).astype(np.float32)

In [ ]:
def build_embedding_caches():
    dev_ready = cache_ready(EMBED_CACHE_DEV, 'dev')
    hidden_ready = cache_ready(EMBED_CACHE_HIDDEN, 'hidden')
    if dev_ready and hidden_ready:
        print('Reusing existing embedding caches.')
        return

    import torch
    from transformers import AutoModel, AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU to compute embeddings.')
    device = torch.device('cuda:0')
    tokenizer = AutoTokenizer.from_pretrained(EMBEDDER_NAME, padding_side='left')
    model = AutoModel.from_pretrained(
        EMBEDDER_NAME, dtype=torch.float16,
        attn_implementation='sdpa', low_cpu_mem_usage=True).to(device).eval()
    try:
        if not dev_ready:
            docs = encode_embeddings([p['text'] for p in corpus], 'Corpus', model, tokenizer, device)
            dev = encode_embeddings([f'Instruct: {QUERY_INSTRUCTION}\nQuery: {q["question"]}'
                                     for q in questions['dev']], 'Dev', model, tokenizer, device)
            np.savez_compressed(EMBED_CACHE_DEV,
                doc_ids=np.asarray([p['doc_id'] for p in corpus]), doc_vectors=docs,
                qids=np.asarray([q['qid'] for q in questions['dev']]), query_vectors=dev,
                checkpoint=np.asarray(EMBEDDER_NAME), max_length=np.asarray(EMBED_MAX_LENGTH))
        if not hidden_ready:
            hidden = encode_embeddings([f'Instruct: {QUERY_INSTRUCTION}\nQuery: {q["question"]}'
                                        for q in questions['hidden']], 'Hidden', model, tokenizer, device)
            np.savez_compressed(EMBED_CACHE_HIDDEN,
                qids=np.asarray([q['qid'] for q in questions['hidden']]),
                query_vectors=hidden, checkpoint=np.asarray(EMBEDDER_NAME),
                max_length=np.asarray(EMBED_MAX_LENGTH))
    finally:
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
build_embedding_caches()

## 6. Dense retrieval

Use normalized Qwen embeddings and exact cosine similarity from the shared source.

In [ ]:
with np.load(EMBED_CACHE_DEV, allow_pickle=False) as data:
    doc_ids = data['doc_ids'].astype(str)
    doc_vectors = data['doc_vectors'].astype(np.float32)
    query_vectors_dev = data['query_vectors'].astype(np.float32)
    qids_dev = data['qids'].astype(str).tolist()

with np.load(EMBED_CACHE_HIDDEN, allow_pickle=False) as data:
    query_vectors_hidden = data['query_vectors'].astype(np.float32)
    qids_hidden = data['qids'].astype(str).tolist()

cosine_rank = partial(core_cosine_rank, doc_vectors=doc_vectors, doc_ids=doc_ids)

In [ ]:
dense_full = {
    'dev': {qid: cosine_rank(vector, MAX_CANDIDATES)
            for qid, vector in zip(qids_dev, query_vectors_dev)},
    'hidden': {qid: cosine_rank(vector, MAX_CANDIDATES)
               for qid, vector in zip(qids_hidden, query_vectors_hidden)},
}
print('Dense rankings ready.')

## 7. Tune the hybrid retriever on Dev

Sweep Dense and BM25 candidate depths (10, 20, 50). Select the best HitAll@10, then Recall@10. No hidden labels are used.

In [ ]:
def retrieval_metrics(ranks, k=10):
    recalls, hitalls = [], []
    for question in questions['dev']:
        qid = question['qid']
        gold = set(labels_dev[qid]['gold_doc_ids'])
        retrieved = set(ranks[qid][:k])
        recalls.append(len(gold & retrieved) / len(gold))
        hitalls.append(int(gold <= retrieved))
    return {'recall10': 100 * float(np.mean(recalls)),
            'hitall10': 100 * float(np.mean(hitalls))}

sparse_full_dev = {
    q['qid']: bm25_indexes['passage'].search(q['question'], k=MAX_CANDIDATES)
    for q in questions['dev']
}

In [ ]:
def fuse_at_depth(split, sparse_by_qid, dense_k, bm25_k):
    return {
        q['qid']: rrf_fusion([
            dense_full[split][q['qid']][:dense_k],
            sparse_by_qid[q['qid']][:bm25_k],
        ], k=RETRIEVED_K, rank_constant=RRF_K)
        for q in questions[split]
    }

tuning = []
for dense_k in CANDIDATE_OPTIONS:
    for bm25_k in CANDIDATE_OPTIONS:
        ranks = fuse_at_depth('dev', sparse_full_dev, dense_k, bm25_k)
        tuning.append({'dense_k': dense_k, 'bm25_k': bm25_k,
                       **retrieval_metrics(ranks)})

tuning.sort(key=lambda row: (-row['hitall10'], -row['recall10'],
                             row['dense_k'] + row['bm25_k'],
                             row['dense_k'], row['bm25_k']))
CHOSEN_DENSE_K = int(CFG['retrieval']['dense_candidates'])
CHOSEN_BM25_K = int(CFG['retrieval']['bm25_candidates'])
print('Best dev sweep:', tuning[0]['dense_k'], tuning[0]['bm25_k'])
print('Configured candidate depths:', CHOSEN_DENSE_K, CHOSEN_BM25_K)
print('Dev retrieval:', retrieval_metrics(
    fuse_at_depth('dev', sparse_full_dev, CHOSEN_DENSE_K, CHOSEN_BM25_K)))


In [ ]:
with (REPORT_DIR / 'retrieval_candidate_sweep.json').open('w', encoding='utf-8') as file:
    json.dump({
        'criterion': 'maximize HitAll@10, then Recall@10, then minimize candidate depths',
        'selected': {'dense_k': CHOSEN_DENSE_K, 'bm25_k': CHOSEN_BM25_K},
        'grid': tuning,
    }, file, indent=2)

## 8. Build the retrieval variants

Create the fair Dense baseline, passage-level Hybrid, title/sentence variants, and a gold-masked Dev run.

In [ ]:
rankings = {}
for split in ('dev', 'hidden'):
    rankings[(split, 'dense_fair')] = {
        q['qid']: dense_full[split][q['qid']][:RETRIEVED_K]
        for q in questions[split]
    }
    sparse = sparse_full_dev if split == 'dev' else {
        q['qid']: bm25_indexes['passage'].search(q['question'], k=CHOSEN_BM25_K)
        for q in questions['hidden']
    }
    rankings[(split, 'hybrid')] = fuse_at_depth(
        split, sparse, CHOSEN_DENSE_K, CHOSEN_BM25_K)

In [ ]:
for mode in ('title', 'sentence'):
    sparse = {
        q['qid']: bm25_indexes[mode].search(q['question'], k=CHOSEN_BM25_K)
        for q in questions['dev']
    }
    rankings[('dev', mode)] = fuse_at_depth(
        'dev', sparse, CHOSEN_DENSE_K, CHOSEN_BM25_K)

In [ ]:
masked = {}
for question, vector in zip(questions['dev'], query_vectors_dev):
    qid = question['qid']
    excluded = set(labels_dev[qid]['gold_doc_ids'])
    dense = cosine_rank(vector, CHOSEN_DENSE_K, excluded=excluded)
    sparse = bm25_indexes['passage'].search(
        question['question'], k=CHOSEN_BM25_K, exclude_ids=excluded)
    retrieved = rrf_fusion([dense, sparse], k=RETRIEVED_K, rank_constant=RRF_K)
    if excluded.intersection(retrieved):
        raise ValueError(f'Gold-masked retrieval leaked a gold document: {qid}')
    masked[qid] = retrieved

rankings[('dev', 'masked')] = masked
print('Rankings created for Dense, Hybrid, Title, Sentence, and Gold-Masked.')

## 9. Context and prompt

The context has a strict 1,800-token budget. Answers must cite only documents packed into the prompt.

In [ ]:
pack_context = partial(core_pack_context, corpus_by_id=corpus_by_id)

PROMPT_SYSTEM = CFG['generation']['prompt_text']

def chat_messages(question, context):
    return [{'role':'system','content': PROMPT_SYSTEM},
            {'role':'user','content': f'CONTEXT:\n{context}\n\nQUESTION:\n{question}\n\nJSON:'}]


## 10. Load the language model

Qwen3.5-9B is loaded in 4-bit NF4 precision. Generation is greedy for all systems.

In [ ]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, Qwen3_5ForCausalLM
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU before loading Qwen.')
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
print('CUDA GPUs:', [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=LLM_QUANTIZATION,
                                bnb_4bit_use_double_quant=True,
                                bnb_4bit_compute_dtype=getattr(torch, LLM_DTYPE))
llm_model = Qwen3_5ForCausalLM.from_pretrained(
    LLM_NAME, quantization_config=bnb_config, device_map='auto', dtype=getattr(torch, LLM_DTYPE),
    attn_implementation='sdpa', low_cpu_mem_usage=True
).eval()
print('Qwen LLM loaded:', LLM_NAME)


## 11. Grounded generation

Pack retrieved passages, generate a JSON answer, and validate citations against the packed documents.

In [ ]:
def prompt_ids(question, context):
    rendered = llm_tokenizer.apply_chat_template(
        chat_messages(question, context), tokenize=False,
        add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    return llm_tokenizer(rendered, return_tensors='pt', add_special_tokens=False)

def grounded_generate(question, ranked_ids):
    context, packed = pack_context(ranked_ids, llm_tokenizer, CONTEXT_TOKEN_BUDGET)
    if not packed:
        return {'answer':'', 'cited':[], 'abstained':True}, [], ''
    tokens = prompt_ids(question,context)
    while packed and tokens['input_ids'].shape[-1] > MAX_INPUT_TOKENS:
        packed.pop()
        context, packed = pack_context(packed, llm_tokenizer, CONTEXT_TOKEN_BUDGET)
        tokens = prompt_ids(question,context)
    if not packed:
        return {'answer':'', 'cited':[], 'abstained':True}, [], ''
    input_len = tokens['input_ids'].shape[-1]
    tokens = tokens.to(llm_model.device)
    with torch.inference_mode():
        response = llm_model.generate(
            **tokens,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    raw = llm_tokenizer.decode(response[0][input_len:], skip_special_tokens=True)
    return parse_grounded_output(raw, packed), packed, raw

print('Generator ready: do_sample=False, enable_thinking=ENABLE_THINKING, max_new_tokens=',MAX_NEW_TOKENS)


## 12. Save predictions and run settings

Each system writes `predictions.jsonl` and `run_config.yaml`. Existing complete rows are reused.

In [ ]:
def config_for(system, split):
    run_key = f'{split}_{system}'
    retrieval = dict(CFG['runs'][run_key])
    dense_only = system == 'dense_fair'
    retrieval.update({
        'dense_top_k': RETRIEVED_K if dense_only else CHOSEN_DENSE_K,
        'bm25_top_k': 0 if dense_only else CHOSEN_BM25_K,
        'retrieved_k': RETRIEVED_K,
        'bm25_k1': BM25_K1, 'bm25_b': BM25_B,
        'rrf_k': None if dense_only else RRF_K,
    })
    return {
        'system_name': f'phase2_fair_{system}', 'phase': CFG['phase'], 'split': split,
        'description': ('New exact Dense baseline (fair) with common generation settings'
                        if dense_only else
                        'Qwen3 Dense embedding + handwritten BM25 + RRF, common grounded generation'),
        'models': CFG['models'],
        'settings': {
            'retrieval': retrieval,
            'max_context_tokens': CONTEXT_TOKEN_BUDGET,
            'max_input_tokens': MAX_INPUT_TOKENS,
            'max_new_tokens': MAX_NEW_TOKENS,
            'embedding_max_length': EMBED_MAX_LENGTH,
            'enable_thinking': ENABLE_THINKING,
            'prompt_text': PROMPT_SYSTEM,
            'context_template': CFG['generation']['context_template'],
            'candidate_tuning': 'dev labels retrieval-only',
            'baseline_origin': 'new dense_fair/dev produced by THIS notebook',
        },
        'command': 'Kaggle notebook: run_system(system,split)',
    }


In [ ]:
def run_system(system, split):
    selected = questions[split] if LIMIT_QUESTIONS is None else questions[split][:LIMIT_QUESTIONS]
    parent = OUTPUT_DIR / ('smoke' if LIMIT_QUESTIONS is not None else 'full') / split / system
    parent.mkdir(parents=True, exist_ok=True)
    cfg_file = parent / 'run_config.yaml'
    pred_file = parent / 'predictions.jsonl'
    cfg = config_for(system, split)

    if cfg_file.exists() and yaml.safe_load(cfg_file.read_text(encoding='utf-8')) != cfg:
        raise ValueError('An older run has different settings. Use a new output directory.')
    cfg_file.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')

    previous = read_jsonl(pred_file) if pred_file.exists() else []
    seen = {row['qid'] for row in previous}
    if len(seen) != len(previous):
        raise ValueError('Duplicate question IDs in saved predictions.')

    with pred_file.open('a', encoding='utf-8') as out:
        for index, question in enumerate(selected, 1):
            qid = question['qid']
            if qid in seen:
                continue
            retrieved = rankings[(split, system)][qid]
            answer, packed, _ = grounded_generate(question['question'], retrieved)
            row = {'qid': qid, 'answer': answer['answer'], 'retrieved': retrieved,
                   'cited': answer['cited'], 'abstained': answer['abstained']}
            out.write(json.dumps(row, ensure_ascii=False) + '\n')
            out.flush()
            if index % 50 == 0 or index == len(selected):
                print(f'{split}/{system}: {index}/{len(selected)}')
    return pred_file

## 13. Run the Dev experiments

Dense and Hybrid use the same model, prompt, generation settings, and context budget.

In [ ]:
dense_fair_dev_path = run_system('dense_fair', 'dev')
hybrid_dev_path = run_system('hybrid', 'dev')

In [ ]:
if GENERATE_GRANULARITY_ANSWERS:
    title_dev_path = run_system('title', 'dev')
    sentence_dev_path = run_system('sentence', 'dev')
else:
    title_dev_path = sentence_dev_path = None

masked_dev_path = run_system('masked', 'dev')

## 14. Run Hybrid on Hidden

The hidden split uses only the candidate depths selected on Dev.

In [ ]:
hybrid_hidden_path = run_system('hybrid', 'hidden')

## 15. Official evaluation

Save full reports from the course `score.py`, including a paired Hybrid-vs-Dense comparison.

In [ ]:
from subprocess import run
scorer = SOURCE_DIR/'score.py'
mode_dir = 'smoke' if LIMIT_QUESTIONS is not None else 'full'

def score_one(path, split, compare=None, label=None):
    args=[sys.executable,str(scorer),str(path),'--pack',str(PACK),'--split',split]
    if split=='hidden':
        args.append('--validate-only')
    if LIMIT_QUESTIONS is not None and split=='dev':
        args.append('--partial')
    if compare is not None and LIMIT_QUESTIONS is None:
        args.extend(['--compare',str(compare)])
    p=run(args,capture_output=True,text=True)
    result=p.stdout+'\n'+p.stderr
    name=label or path.parent.name
    target=REPORT_DIR/f'{mode_dir}_{split}_{name}.txt'
    target.write_text(result,encoding='utf-8')
    summary = []
    for metric in ('em', 'f1', 'recall@10', 'hitall@10', 'abstained'):
        match = re.search(r'^  ' + re.escape(metric) + r'\s+([0-9.]+)', result, re.MULTILINE)
        if match:
            summary.append(f'{metric}={match.group(1)}')
    print(f'{name}: ' + ', '.join(summary))
    if p.returncode:
        raise RuntimeError(f'score.py failed: {target}, exit={p.returncode}')

In [ ]:
score_one(dense_fair_dev_path,'dev',label='dense_fair')
score_one(hybrid_dev_path,'dev',compare=dense_fair_dev_path,label='hybrid_minus_dense_fair')
if LIMIT_QUESTIONS is None:
    score_one(hybrid_hidden_path,'hidden',label='hybrid_hidden')
else:
    print('Partial hidden rows: skipping complete-split validation.')
if title_dev_path:
    score_one(title_dev_path,'dev',label='title')
if sentence_dev_path:
    score_one(sentence_dev_path,'dev',label='sentence')
score_one(masked_dev_path,'dev',label='masked')
print('Official score reports saved in:', REPORT_DIR)

## 16. Compare question types

Summarize EM, F1, Recall@10, HitAll@10, and abstention for each Dev question type.

In [ ]:
import pandas as pd


def read_score_summary(path, system):
    rows = []
    question_type = None
    metrics = {}

    def save_row():
        if question_type and metrics:
            rows.append({'System': system, 'Type': question_type, **metrics})

    for line in path.read_text(encoding='utf-8').splitlines():
        if line.startswith('paired difference:'):
            break
        if line.startswith('overall (') or line.startswith('type: '):
            save_row()
            question_type = ('overall' if line.startswith('overall')
                             else line.split('(')[0].removeprefix('type: ').strip())
            metrics = {}
        else:
            fields = line.split()
            if len(fields) > 1 and fields[0] in ('em', 'f1', 'recall@10', 'hitall@10', 'abstained'):
                metrics[fields[0]] = float(fields[1])
    save_row()
    return rows

In [ ]:
reports = {
    'Dense': 'dense_fair',
    'Hybrid': 'hybrid_minus_dense_fair',
    'Title': 'title',
    'Sentence': 'sentence',
    'Gold-Masked': 'masked',
}
summary_rows = []
for system, label in reports.items():
    report_path = REPORT_DIR / f'{mode_dir}_dev_{label}.txt'
    if report_path.exists():
        summary_rows.extend(read_score_summary(report_path, system))

score_table = pd.DataFrame(summary_rows)
if not score_table.empty:
    display(score_table.round(1))

## 17. Gold-masked analysis

Count abstentions, lucky correct answers, and unsupported answers when gold documents are excluded.

In [ ]:
from importlib.util import spec_from_file_location, module_from_spec
spec = spec_from_file_location('course_score', scorer)
course_score = module_from_spec(spec)
spec.loader.exec_module(course_score)
mask_rows = read_jsonl(masked_dev_path)
counts = Counter()
for row in mask_rows:
    if row['abstained']:
        counts['abstained'] += 1
    else:
        lab = labels_dev[row['qid']]
        accepted = [lab['answer']] + lab.get('aliases', [])
        if any(course_score.normalize(row['answer']) == course_score.normalize(x) for x in accepted):
            counts['lucky_correct'] += 1
        else:
            counts['wrong_or_unsupported'] += 1
print('Gold-masked breakdown:',dict(counts),'(n =',len(mask_rows),')')

## 18. Export submission outputs

Pack predictions, run configurations, and official evaluation reports into one ZIP file.

In [ ]:
final_zip = ROOT / ('Phase2_Fair_Smoke_Outputs.zip' if LIMIT_QUESTIONS is not None
                    else 'Phase2_Fair_Submission_Outputs.zip')
with zipfile.ZipFile(final_zip,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    system_base=OUTPUT_DIR/mode_dir
    for path in system_base.rglob('*'):
        if path.is_file():
            archive.write(path,Path('runs')/path.relative_to(system_base))
    for path in REPORT_DIR.glob('*'):
        if path.is_file() and path.suffix in ('.txt','.json'):
            archive.write(path,Path('reports')/path.name)
print('Output archive:',final_zip,'size KB:',round(final_zip.stat().st_size/1024))